# L0 vs F1 / MCC curves across SAE architectures

For each distribution, shows the best F1 (or MCC) achievable at each L0 value, per SAE architecture.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

datasets = {
    "BatchTopK": "data/batch_total/results.parquet",
    "Matryoshka": "data/matryoshka_total/results.parquet",
    "MatchingPursuit": "data/mp_total/results.parquet",
    "Standard": "data/standard_total/results.parquet",
}

dfs = []
for arch, path in datasets.items():
    d = pd.read_parquet(path).reset_index()
    dfs.append(d)

raw = pd.concat(dfs, ignore_index=True)
raw = raw.rename(columns={"f1_score": "f1"})

# For each (benchmark, sae_type, sae_l0) keep the best F1 and MCC.
# (Matryoshka has multiple configs per integer L0 level.)
best = (
    raw.groupby(["benchmark", "sae_type", "sae_l0"], as_index=False)
    .agg(f1=("f1", "max"), mcc=("mcc", "max"))
    .sort_values(["benchmark", "sae_type", "sae_l0"])
    .reset_index(drop=True)
)

benchmarks = sorted(best["benchmark"].unique())
arch_order = ["BatchTopK", "Matryoshka", "MatchingPursuit", "Standard"]
colors = {
    "BatchTopK": "#636EFA",
    "Matryoshka": "#EF553B",
    "MatchingPursuit": "#00CC96",
    "Standard": "#AB63FA",
}

print(f"Benchmarks: {benchmarks}")
print(f"Architectures: {best['sae_type'].unique().tolist()}")
print(f"Total rows after groupby: {len(best)}")

In [ ]:
def make_l0_plot(metric: str, y_label: str) -> go.Figure:
    n_cols = 4
    n_rows = 2
    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=benchmarks,
        shared_xaxes=False,
        shared_yaxes=False,
        horizontal_spacing=0.07,
        vertical_spacing=0.14,
    )

    shown_legends = set()
    for idx, benchmark in enumerate(benchmarks):
        row = idx // n_cols + 1
        col = idx % n_cols + 1
        sub = best[best["benchmark"] == benchmark]

        for arch in arch_order:
            arch_data = sub[sub["sae_type"] == arch].sort_values("sae_l0")
            if arch_data.empty:
                continue
            show_legend = arch not in shown_legends
            shown_legends.add(arch)
            fig.add_trace(
                go.Scatter(
                    x=arch_data["sae_l0"],
                    y=arch_data[metric],
                    mode="lines+markers",
                    name=arch,
                    line=dict(color=colors[arch]),
                    marker=dict(size=6, color=colors[arch]),
                    legendgroup=arch,
                    showlegend=show_legend,
                ),
                row=row,
                col=col,
            )

        fig.update_xaxes(title_text="L0", row=row, col=col)
        fig.update_yaxes(title_text=y_label if col == 1 else "", row=row, col=col)

    fig.update_layout(
        height=600,
        width=1400,
        title_text=f"Best {y_label} at each L0, per SAE architecture",
        legend=dict(orientation="h", yanchor="bottom", y=1.04, xanchor="right", x=1),
    )
    return fig

In [ ]:
# Graph 1: L0 vs F1
fig_f1 = make_l0_plot("f1", "F1")
fig_f1.show()

In [ ]:
# Graph 2: L0 vs MCC
fig_mcc = make_l0_plot("mcc", "MCC")
fig_mcc.show()